<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/AI_Powered_Scientific_Knowledge_Graph_of_Global_Science8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
The goal of this project is to build an AI-driven Science Knowledge Graph and Intelligence System. The system will ingest metadata for over 5,000 research papers from major scientific databases (arXiv, Crossref, OpenAlex), extract key entities and relationships using NLP, and construct a multi-dimensional knowledge graph. We will perform graph analysis to identify influential researchers and institutions, use topic modeling to detect emerging research trends, and create an interactive dashboard for visualization. Finally, a retrieval-augmented generation (RAG) interface will be implemented to allow natural language querying of the research data. All structured data will be exported into specialized CSV files and a comprehensive Excel workbook.

## Environment Setup and API Integration

### Subtask:
Install required Python libraries and implement functions to query arXiv, Crossref, and OpenAlex APIs.


**Reasoning**:
Installing the required libraries is the first step to ensure all subsequent code runs correctly.



In [1]:
%%capture
!pip install networkx pyvis bertopic sentence-transformers pandas requests openpyxl
import pandas as pd
import requests
import networkx as nx
print('Libraries installed and imported successfully.')

**Reasoning**:
I will define the Python functions to query arXiv, Crossref, and OpenAlex. These functions will handle HTTP requests and parse the responses into a common format for later processing.



In [2]:
def fetch_arxiv_papers(query, max_results=100):
    url = f'http://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}'
    response = requests.get(url)
    # ArXiv returns XML; we will use a basic parser approach or feed into a more complex one later.
    return response.text

def fetch_crossref_papers(query, rows=100):
    url = f'https://api.crossref.org/works?query={query}&rows={rows}'
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    return None

def fetch_openalex_papers(query, per_page=100):
    url = f'https://api.openalex.org/works?search={query}&per_page={per_page}'
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    return None

print('API fetching functions for ArXiv, Crossref, and OpenAlex have been defined.')

API fetching functions for ArXiv, Crossref, and OpenAlex have been defined.


## Data Collection and Normalization

### Subtask:
Fetch 5,000+ papers using the defined APIs, deduplicate the records, and normalize the schemas into a unified pandas DataFrame. Export the results to CSV files.


**Reasoning**:
I will start by implementing the data collection logic. Since parsing XML from arXiv and handling diverse JSON from multiple sources is complex, I will first define a robust parsing and fetching routine to gather the raw data into a list of dictionaries.



In [3]:
import xml.etree.ElementTree as ET
import time

def parse_arxiv_xml(xml_data):
    papers = []
    try:
        root = ET.fromstring(xml_data)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}
        for entry in root.findall('atom:entry', ns):
            title = entry.find('atom:title', ns).text.strip().replace('\n', ' ')
            authors = [author.find('atom:name', ns).text for author in entry.findall('atom:author', ns)]
            published = entry.find('atom:published', ns).text[:4]
            link = entry.find('atom:id', ns).text
            papers.append({'title': title, 'authors': authors, 'year': published, 'id': link, 'source': 'arXiv', 'institution': 'Unknown'})
    except Exception as e:
        print(f'Error parsing arXiv XML: {e}')
    return papers

def collect_data(queries, limit_per_source=1700):
    all_papers = []
    for query in queries:
        print(f'Fetching data for: {query}...')
        # ArXiv
        arxiv_raw = fetch_arxiv_papers(query, max_results=500)
        all_papers.extend(parse_arxiv_xml(arxiv_raw))

        # Crossref
        cr_data = fetch_crossref_papers(query, rows=500)
        if cr_data and 'message' in cr_data:
            for item in cr_data['message']['items']:
                all_papers.append({
                    'title': item.get('title', [''])[0],
                    'authors': [f"{a.get('given', '')} {a.get('family', '')}".strip() for a in item.get('author', [])],
                    'year': str(item.get('published-print', item.get('created', {})).get('date-parts', [[0]])[0][0]),
                    'id': item.get('DOI', ''),
                    'source': 'Crossref',
                    'institution': 'Various'
                })

        # OpenAlex
        oa_data = fetch_openalex_papers(query, per_page=200)
        if oa_data and 'results' in oa_data:
            for res in oa_data['results']:
                all_papers.append({
                    'title': res.get('display_name', ''),
                    'authors': [au.get('author', {}).get('display_name', '') for au in res.get('authorships', [])],
                    'year': str(res.get('publication_year', '')),
                    'id': res.get('id', ''),
                    'source': 'OpenAlex',
                    'institution': res.get('authorships', [{}])[0].get('institutions', [{}])[0].get('display_name', 'Unknown') if res.get('authorships') else 'Unknown'
                })
        time.sleep(1) # Rate limiting respect

    return pd.DataFrame(all_papers)

queries = ['artificial intelligence', 'climate change', 'genomics', 'blockchain', 'robotics']
df_raw = collect_data(queries)
print(f'Collected {len(df_raw)} raw records.')
df_raw.head()

Fetching data for: artificial intelligence...


IndexError: list index out of range

**Reasoning**:
I will fix the IndexError by implementing a safer extraction method for the institution field in the OpenAlex processing loop. Specifically, I will check if the 'institutions' list is present and non-empty before accessing its first element. I will also provide the full corrected code for the data collection and normalization subtask.



In [4]:
import xml.etree.ElementTree as ET
import time

def parse_arxiv_xml(xml_data):
    papers = []
    try:
        root = ET.fromstring(xml_data)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}
        for entry in root.findall('atom:entry', ns):
            title_elem = entry.find('atom:title', ns)
            title = title_elem.text.strip().replace('\n', ' ') if title_elem is not None else 'Untitled'
            authors = [author.find('atom:name', ns).text for author in entry.findall('atom:author', ns)]
            pub_elem = entry.find('atom:published', ns)
            published = pub_elem.text[:4] if pub_elem is not None else '0000'
            id_elem = entry.find('atom:id', ns)
            link = id_elem.text if id_elem is not None else ''
            papers.append({'title': title, 'authors': authors, 'year': published, 'id': link, 'source': 'arXiv', 'institution': 'Unknown'})
    except Exception as e:
        print(f'Error parsing arXiv XML: {e}')
    return papers

def collect_data(queries, limit_per_source=1700):
    all_papers = []
    for query in queries:
        print(f'Fetching data for: {query}...')
        # ArXiv
        arxiv_raw = fetch_arxiv_papers(query, max_results=500)
        all_papers.extend(parse_arxiv_xml(arxiv_raw))

        # Crossref
        cr_data = fetch_crossref_papers(query, rows=500)
        if cr_data and 'message' in cr_data:
            for item in cr_data['message']['items']:
                all_papers.append({
                    'title': item.get('title', ['Untitled'])[0],
                    'authors': [f"{a.get('given', '')} {a.get('family', '')}".strip() for a in item.get('author', [])],
                    'year': str(item.get('published-print', item.get('created', {})).get('date-parts', [[0]])[0][0]),
                    'id': item.get('DOI', ''),
                    'source': 'Crossref',
                    'institution': 'Various'
                })

        # OpenAlex
        oa_data = fetch_openalex_papers(query, per_page=200)
        if oa_data and 'results' in oa_data:
            for res in oa_data['results']:
                authorships = res.get('authorships', [])
                inst_name = 'Unknown'
                if authorships:
                    first_author_insts = authorships[0].get('institutions', [])
                    if first_author_insts:
                        inst_name = first_author_insts[0].get('display_name', 'Unknown')

                all_papers.append({
                    'title': res.get('display_name', 'Untitled'),
                    'authors': [au.get('author', {}).get('display_name', '') for au in authorships],
                    'year': str(res.get('publication_year', '0000')),
                    'id': res.get('id', ''),
                    'source': 'OpenAlex',
                    'institution': inst_name
                })
        time.sleep(1) # Rate limiting respect

    return pd.DataFrame(all_papers)

queries = ['artificial intelligence', 'climate change', 'genomics', 'blockchain', 'robotics']
df_raw = collect_data(queries)

# Deduplication
df_raw['title_clean'] = df_raw['title'].str.lower().str.replace(r'[^a-z0-9]', '', regex=True)
df_clean = df_raw.drop_duplicates(subset=['title_clean']).drop(columns=['title_clean'])

print(f'Collected {len(df_raw)} raw records.')
print(f'Deduplicated to {len(df_clean)} unique records.')
df_clean.to_csv('papers_raw.csv', index=False)
df_clean.head()

Fetching data for: artificial intelligence...
Fetching data for: climate change...
Fetching data for: genomics...
Fetching data for: blockchain...
Fetching data for: robotics...
Collected 6000 raw records.
Deduplicated to 5048 unique records.


,title,authors,year,id,source,institution
0,"The Artificial Scientist: Logicist, Emergentis...","[Michael Timothy Bennett, Yoshihiro Maruyama]",2021,http://arxiv.org/abs/2110.01831v1,arXiv,Unknown
1,"Compression, The Fermi Paradox and Artificial ...",[Michael Timothy Bennett],2021,http://arxiv.org/abs/2110.01835v1,arXiv,Unknown
2,Creative Problem Solving in Artificially Intel...,"[Evana Gizzi, Lakshmi Nair, Sonia Chernova, Ji...",2022,http://arxiv.org/abs/2204.10358v1,arXiv,Unknown
3,A Review on Explainable Artificial Intelligenc...,"[Subrato Bharati, M. Rubaiyat Hossain Mondal, ...",2023,http://arxiv.org/abs/2304.04780v1,arXiv,Unknown
4,Artificial Intelligence Framework for Simulati...,"[Casey C. Bennett, Kris Hauser]",2013,http://arxiv.org/abs/1301.2158v1,arXiv,Unknown


**Reasoning**:
I will now process the unified DataFrame to extract and normalize author and institution information into separate CSV files. This involves exploding the 'authors' list and mapping institutions to their respective papers and authors to create authors.csv and institutions.csv.



In [5]:
import pandas as pd

# 1. Generate Authors CSV
# Explode the authors list to have one row per author-paper pair
authors_df = df_clean.explode('authors').rename(columns={'authors': 'author_name'})
# Remove rows with empty author names
authors_df = authors_df[authors_df['author_name'].str.strip() != '']
# Keep relevant columns for authors.csv
authors_csv = authors_df[['author_name', 'id', 'year', 'source', 'institution']].copy()
authors_csv.to_csv('authors.csv', index=False)

# 2. Generate Institutions CSV
# Identify unique institutions and link them to papers/sources
institutions_df = df_clean[['institution', 'id', 'source', 'year']].copy()
# Filter out 'Unknown' or 'Various' if we want a cleaner list, but for now we keep them to maintain relationships
institutions_df.to_csv('institutions.csv', index=False)

print(f'Successfully exported authors.csv with {len(authors_csv)} entries.')
print(f'Successfully exported institutions.csv with {len(institutions_df)} entries.')
display(authors_csv.head())
display(institutions_df.head())

Successfully exported authors.csv with 20506 entries.
Successfully exported institutions.csv with 5048 entries.


,author_name,id,year,source,institution
0,Michael Timothy Bennett,http://arxiv.org/abs/2110.01831v1,2021,arXiv,Unknown
0,Yoshihiro Maruyama,http://arxiv.org/abs/2110.01831v1,2021,arXiv,Unknown
1,Michael Timothy Bennett,http://arxiv.org/abs/2110.01835v1,2021,arXiv,Unknown
2,Evana Gizzi,http://arxiv.org/abs/2204.10358v1,2022,arXiv,Unknown
2,Lakshmi Nair,http://arxiv.org/abs/2204.10358v1,2022,arXiv,Unknown


,institution,id,source,year
0,Unknown,http://arxiv.org/abs/2110.01831v1,arXiv,2021
1,Unknown,http://arxiv.org/abs/2110.01835v1,arXiv,2021
2,Unknown,http://arxiv.org/abs/2204.10358v1,arXiv,2022
3,Unknown,http://arxiv.org/abs/2304.04780v1,arXiv,2023
4,Unknown,http://arxiv.org/abs/1301.2158v1,arXiv,2013


## AI Entity and Relationship Extraction

### Subtask:
Use NLP techniques to extract research concepts and define relationships (Paper-Author, Author-Institution, Concept-Concept) from the collected metadata.


**Reasoning**:
I will begin by extracting concepts from the paper titles using a basic keyword extraction approach and then construct the entity and relationship dataframes.



In [6]:
import pandas as pd
import re

# 1. Extract Concepts from Titles (Simplified Entity Extraction)
keywords = ['machine learning', 'deep learning', 'climate', 'genome', 'crispr', 'blockchain', 'neural network', 'robot', 'automation', 'intelligence']
def extract_concepts(title):
    found = []
    for kw in keywords:
        if re.search(r'\b' + kw + r'\b', title.lower()):
            found.append(kw.title())
    return found

df_clean['concepts'] = df_clean['title'].apply(extract_concepts)

# 2. Build Entities CSV
author_entities = pd.DataFrame({'entity_id': authors_csv['author_name'].unique(), 'type': 'Author'})
inst_entities = pd.DataFrame({'entity_id': institutions_df['institution'].unique(), 'type': 'Institution'})
concept_list = [item for sublist in df_clean['concepts'] for item in sublist]
concept_entities = pd.DataFrame({'entity_id': list(set(concept_list)), 'type': 'Concept'})
paper_entities = pd.DataFrame({'entity_id': df_clean['id'], 'type': 'Paper'})

entities_df = pd.concat([author_entities, inst_entities, concept_entities, paper_entities], ignore_index=True)
entities_df.to_csv('entities.csv', index=False)

# 3. Build Relationships CSV
rels = []

# Paper -> Author
for _, row in authors_csv.iterrows():
    rels.append({'source': row['id'], 'target': row['author_name'], 'type': 'AUTHORED_BY'})

# Author -> Institution
for _, row in authors_csv.iterrows():
    if row['institution'] != 'Unknown':
        rels.append({'source': row['author_name'], 'target': row['institution'], 'type': 'AFFILIATED_WITH'})

# Paper -> Concept
for _, row in df_clean.iterrows():
    for concept in row['concepts']:
        rels.append({'source': row['id'], 'target': concept, 'type': 'HAS_CONCEPT'})

relationships_df = pd.DataFrame(rels)
relationships_df.to_csv('relationships.csv', index=False)

print(f'Extracted {len(entities_df)} entities and {len(relationships_df)} relationships.')
print(entities_df['type'].value_counts())
display(relationships_df.head())

Extracted 21974 entities and 32779 relationships.
type
Author         16429
Paper           5048
Institution      487
Concept           10
Name: count, dtype: int64


,source,target,type
0,http://arxiv.org/abs/2110.01831v1,Michael Timothy Bennett,AUTHORED_BY
1,http://arxiv.org/abs/2110.01831v1,Yoshihiro Maruyama,AUTHORED_BY
2,http://arxiv.org/abs/2110.01835v1,Michael Timothy Bennett,AUTHORED_BY
3,http://arxiv.org/abs/2204.10358v1,Evana Gizzi,AUTHORED_BY
4,http://arxiv.org/abs/2204.10358v1,Lakshmi Nair,AUTHORED_BY


## Knowledge Graph Construction and Metrics

### Subtask:
Build the knowledge graph using NetworkX, calculate centrality measures, and perform community detection.


**Reasoning**:
I will load the entity and relationship data into a NetworkX graph, calculate key centrality metrics, and use the community detection algorithm to group nodes.



In [7]:
import networkx as nx
import pandas as pd
from networkx.algorithms import community

# 1. Initialize Graph
G = nx.Graph()

# 2. Add Nodes and Edges
for _, row in entities_df.iterrows():
    G.add_node(row['entity_id'], type=row['type'])

for _, row in relationships_df.iterrows():
    G.add_edge(row['source'], row['target'], type=row['type'])

# 3. Calculate Centrality Measures
print('Calculating centrality metrics...')
degree_cent = nx.degree_centrality(G)
betweenness_cent = nx.betweenness_centrality(G, k=500) # k used for approximation on large graphs

# 4. Community Detection (Greedy Modularity Communities)
print('Performing community detection...')
communities = community.greedy_modularity_communities(G)
community_map = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_map[node] = i

# 5. Compile Metrics and Export
metrics_data = []
for node in G.nodes():
    metrics_data.append({
        'entity_id': node,
        'type': G.nodes[node].get('type', 'Unknown'),
        'degree_centrality': degree_cent.get(node, 0),
        'betweenness_centrality': betweenness_cent.get(node, 0),
        'community': community_map.get(node, -1)
    })

graph_metrics_df = pd.DataFrame(metrics_data)
graph_metrics_df.to_csv('graph_metrics.csv', index=False)

print(f'Graph constructed with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.')
print(f'Metrics saved to graph_metrics.csv.')
graph_metrics_df.sort_values(by='degree_centrality', ascending=False).head(10)

Calculating centrality metrics...
Performing community detection...


TypeError: Consider using a tuple, with a priority value that can be compared.

**Reasoning**:
The `TypeError` in `greedy_modularity_communities` often arises from internal comparisons of node objects. I will convert all node IDs to strings explicitly to ensure consistency and retry the metric calculation and community detection.



In [8]:
import networkx as nx
import pandas as pd
from networkx.algorithms import community

# 1. Initialize Graph with consistent string IDs
G = nx.Graph()

# 2. Add Nodes and Edges ensuring IDs are strings
for _, row in entities_df.iterrows():
    node_id = str(row['entity_id'])
    G.add_node(node_id, type=row['type'])

for _, row in relationships_df.iterrows():
    u, v = str(row['source']), str(row['target'])
    # Ensure both nodes exist (especially if entities_df was filtered)
    if u in G and v in G:
        G.add_edge(u, v, type=row['type'])

# 3. Calculate Centrality Measures
print('Calculating centrality metrics...')
degree_cent = nx.degree_centrality(G)
betweenness_cent = nx.betweenness_centrality(G, k=500)

# 4. Community Detection (Greedy Modularity Communities)
print('Performing community detection...')
# Use list() to handle the generator or result consistently
communities = list(community.greedy_modularity_communities(G))

community_map = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_map[node] = i

# 5. Compile Metrics and Export
metrics_data = []
for node in G.nodes():
    metrics_data.append({
        'entity_id': node,
        'type': G.nodes[node].get('type', 'Unknown'),
        'degree_centrality': degree_cent.get(node, 0),
        'betweenness_centrality': betweenness_cent.get(node, 0),
        'community': community_map.get(node, -1)
    })

graph_metrics_df = pd.DataFrame(metrics_data)
graph_metrics_df.to_csv('graph_metrics.csv', index=False)

print(f'Graph constructed with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.')
print('Top nodes by Degree Centrality:')
display(graph_metrics_df.sort_values(by='degree_centrality', ascending=False).head(10))

Calculating centrality metrics...
Performing community detection...
Graph constructed with 21973 nodes and 31551 edges.
Top nodes by Degree Centrality:


,entity_id,type,degree_centrality,betweenness_centrality,community
16430,Various,Institution,0.060850,0.172171,1
16922,Climate,Concept,0.049244,0.280117,0
16924,Blockchain,Concept,0.045877,0.217215,1
16916,Intelligence,Concept,0.039186,0.235480,0
1754,nan,Author,0.031222,0.108446,0
16919,Robot,Concept,0.015838,0.150479,2
16647,Broad Institute,Institution,0.012334,0.021642,3
16921,Genome,Concept,0.010650,0.106780,4
16661,Howard Hughes Medical Institute,Institution,0.008829,0.016758,5
16464,Stanford University,Institution,0.007965,0.023906,2


## Research Trend and Emerging Science Detection

### Subtask:
Implement BERTopic for topic modeling to identify research themes and calculate growth rates/emergence scores.


**Reasoning**:
I will initialize the BERTopic model and perform topic modeling on the cleaned paper titles to extract the main research themes.



In [9]:
from bertopic import BERTopic
import pandas as pd

# Extract titles for modeling
documents = df_clean['title'].tolist()

# Initialize and fit BERTopic
print('Fitting BERTopic model (this may take a moment)...')
topic_model = BERTopic(language='multilingual', calculate_probabilities=False, verbose=True)
topics, probs = topic_model.fit_transform(documents)

# Get topic info
topic_info = topic_model.get_topic_info()
print('Topic modeling complete.')
display(topic_info.head(10))

# Add topics back to the dataframe
df_clean['topic'] = topics

2026-06-22 18:04:22,858 - BERTopic - Embedding - Transforming documents to embeddings.


Fitting BERTopic model (this may take a moment)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]

2026-06-22 18:05:51,833 - BERTopic - Embedding - Completed ✓
2026-06-22 18:05:51,835 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-22 18:06:16,375 - BERTopic - Dimensionality - Completed ✓
2026-06-22 18:06:16,376 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-22 18:06:16,523 - BERTopic - Cluster - Completed ✓
2026-06-22 18:06:16,532 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-22 18:06:16,645 - BERTopic - Representation - Completed ✓


Topic modeling complete.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2022,-1_artificial_intelligence_and_for,"[artificial, intelligence, and, for, of, in, b...",[Artificial intelligence to deep learning: mac...
1,0,110,0_blockchain_technology_blockchains_challenges,"[blockchain, technology, blockchains, challeng...","[The Blockchain, Blockchain 2.0, Blockchain]"
2,1,104,1_robotics_japan_manufacturing_issue,"[robotics, japan, manufacturing, issue, evolut...",[Special Issue on Cutting Edge of Robotics in ...
3,2,76,2_cancer_genomic_characterization_integrated,"[cancer, genomic, characterization, integrated...",[Integrated genomic and molecular characteriza...
4,3,69,3_page_appendix_index_copyright,"[page, appendix, index, copyright, note, title...","[[Title page], Title Page i, Front cover Page]"
5,4,66,4_microbial_bacterial_genomes_genome,"[microbial, bacterial, genomes, genome, coli, ...",[The importance of genome sequence quality to ...
6,5,60,5_genome_assembly_foundation_efficient,"[genome, assembly, foundation, efficient, acce...",[Accelerating Genome Sequence Analysis via Eff...
7,6,59,6_adaptation_resilience_vulnerability_urban,"[adaptation, resilience, vulnerability, urban,...","[Climate Change 2007: Impacts, Adaptation and ..."
8,7,59,7_medicine_healthcare_clinical_artificial,"[medicine, healthcare, clinical, artificial, i...",[Overview of artificial intelligence in medici...
9,8,58,8_learning_reinforcement_robot_deep,"[learning, reinforcement, robot, deep, continu...",[Benchmarking Actor-Critic Deep Reinforcement ...


**Reasoning**:
I will calculate topic growth rates by comparing the frequency of topics in recent years versus earlier years to identify emerging research trends.



In [10]:
import pandas as pd
import numpy as np

# Ensure year is numeric
df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce').fillna(0).astype(int)

# Calculate topic frequency by year
trend_data = df_clean.groupby(['topic', 'year']).size().reset_index(name='count')

# Filter out noise topic (-1)
trend_data = trend_data[trend_data['topic'] != -1]

# Define 'Emergence Score' as growth between two periods (e.g., before 2020 and 2020-present)
recent_cutoff = 2020

topic_growth = []
for topic in trend_data['topic'].unique():
    topic_df = trend_data[trend_data['topic'] == topic]
    before = topic_df[topic_df['year'] < recent_cutoff]['count'].sum()
    after = topic_df[topic_df['year'] >= recent_cutoff]['count'].sum()

    # Simple growth rate calculation
    growth_rate = (after - before) / (before + 1)  # +1 to avoid division by zero
    topic_growth.append({
        'topic': topic,
        'count_before_2020': before,
        'count_after_2020': after,
        'growth_rate': growth_rate
    })

growth_df = pd.DataFrame(topic_growth)

# Merge with topic names from BERTopic
topic_names = topic_model.get_topic_info()[['Topic', 'Name', 'Representation']]
research_trends = growth_df.merge(topic_names, left_on='topic', right_on='Topic').drop(columns=['Topic'])

# Save results
research_trends.to_csv('research_trends.csv', index=False)

print('Research trend analysis complete. Top emerging topics (highest growth):')
display(research_trends.sort_values(by='growth_rate', ascending=False).head(10))

Research trend analysis complete. Top emerging topics (highest growth):


,topic,count_before_2020,count_after_2020,growth_rate,Name,Representation
63,63,0,19,19.000000,63_language_processing_related_natural,"[language, processing, related, natural, reddi..."
92,92,1,12,5.500000,92_ieee_international_conference_2022,"[ieee, international, conference, 2022, vaccin..."
99,99,1,11,5.000000,99_task_heterogeneous_allocation_capabilities,"[task, heterogeneous, allocation, capabilities..."
48,48,3,20,4.250000,48_twitter_media_social_polarisation,"[twitter, media, social, polarisation, youtube..."
29,29,5,27,3.666667,29_machine_learning_deep_model,"[machine, learning, deep, model, forecasting, ..."
86,86,2,12,3.333333,86_quantum_post_secured_classical,"[quantum, post, secured, classical, protocols,..."
90,90,2,11,3.000000,90_downscaling_weather_precipitation_numerical,"[downscaling, weather, precipitation, numerica..."
75,75,3,14,2.750000,75_chairs_message_from_steering,"[chairs, message, from, steering, program, gen..."
19,19,8,31,2.555556,19_security_attacks_cybersecurity_attack,"[security, attacks, cybersecurity, attack, blo..."
103,103,2,9,2.333333,103_traffic_vehicle_transportation_intelligent,"[traffic, vehicle, transportation, intelligent..."


## Intelligence Ranking (Institutions, Authors, Countries)

### Subtask:
Aggregate metadata and graph metrics to rank research entities based on publication volume, source diversity, and network influence.


**Reasoning**:
I will aggregate the paper metadata with the graph metrics to produce the first set of rankings for institutions and authors.



In [11]:
import pandas as pd

# 1. Author Rankings
# Calculate counts and average centrality per author
author_stats = authors_csv.groupby('author_name').agg({
    'id': 'count',
    'source': 'nunique'
}).rename(columns={'id': 'paper_count', 'source': 'source_diversity'})

# Merge with graph metrics
author_metrics = graph_metrics_df[graph_metrics_df['type'] == 'Author'][['entity_id', 'degree_centrality', 'betweenness_centrality', 'community']]
author_rankings = author_stats.merge(author_metrics, left_index=True, right_on='entity_id')

# Final Sort
author_rankings = author_rankings.sort_values(by=['paper_count', 'degree_centrality'], ascending=False)
author_rankings.to_csv('author_rankings.csv', index=False)

# 2. Institution Rankings
# Calculate counts and average centrality of papers associated with institutions
inst_stats = df_clean.groupby('institution').agg({
    'id': 'count',
    'source': 'nunique'
}).rename(columns={'id': 'paper_count', 'source': 'source_diversity'})

inst_metrics = graph_metrics_df[graph_metrics_df['type'] == 'Institution'][['entity_id', 'degree_centrality', 'community']]
institution_rankings = inst_stats.merge(inst_metrics, left_index=True, right_on='entity_id')

# Final Sort (Excluding generic terms like 'Unknown' and 'Various' for top display)
institution_rankings = institution_rankings[~institution_rankings['entity_id'].isin(['Unknown', 'Various'])]
institution_rankings = institution_rankings.sort_values(by=['paper_count', 'degree_centrality'], ascending=False)
institution_rankings.to_csv('institution_rankings.csv', index=False)

print('Intelligence rankings generated.')
print('\nTop 5 Authors:')
display(author_rankings.head(5))
print('\nTop 5 Institutions:')
display(institution_rankings.head(5))

Intelligence rankings generated.

Top 5 Authors:


,paper_count,source_diversity,entity_id,degree_centrality,betweenness_centrality,community
14873,22,3,Robin R. Murphy,0.001092,0.003228,2
1361,14,3,Zibin Zheng,0.000728,0.002715,9
5892,13,1,Onur Mutlu,0.000592,0.000607,4
8092,10,1,Gad Getz,0.000683,0.002307,3
11061,10,2,Ingo Weber,0.000592,0.000108,8



Top 5 Institutions:


,paper_count,source_diversity,entity_id,degree_centrality,community
16485,18,1,Massachusetts Institute of Technology,0.002094,3
16464,15,1,Stanford University,0.007965,2
16433,15,1,Harvard University,0.004597,3
16661,11,1,Howard Hughes Medical Institute,0.008829,5
16857,10,1,Carnegie Mellon University,0.000774,2


# Task
Build an AI-driven Science Knowledge Graph and Intelligence System. The project involves: 1) Ingesting metadata for 5,000+ research papers from arXiv, Crossref, and OpenAlex. 2) Normalizing schemas and extracting entities (Authors, Institutions, Concepts) and relationships (Authored_By, Affiliated_With, Has_Concept) to build a multi-dimensional knowledge graph. 3) Performing graph analysis (centrality, community detection) and BERTopic modeling to detect emerging research trends. 4) Generating intelligence rankings for authors, institutions, and countries. 5) Implementing a RAG interface for natural language querying. 6) Visualizing results in an interactive dashboard and exporting all structured data into a comprehensive multi-sheet Excel workbook.

## Country Intelligence Ranking

### Subtask:
Extract country information from institution names and generate rankings based on research output.


**Reasoning**:
I will use a simple mapping of common city/country keywords found in institution names to identify the country for each record, then aggregate the data to create the rankings.



In [12]:
import pandas as pd

# Heuristic-based country extraction
country_map = {
    'University of': 'Multiple',
    'Institute of Technology': 'Multiple',
    'Stanford': 'USA', 'Harvard': 'USA', 'MIT': 'USA', 'Carnegie Mellon': 'USA', 'Berkeley': 'USA',
    'Oxford': 'UK', 'Cambridge': 'UK', 'Imperial College': 'UK',
    'Tsinghua': 'China', 'Peking': 'China', 'Zhejiang': 'China',
    'ETH Zurich': 'Switzerland', 'EPFL': 'Switzerland',
    'Tokyo': 'Japan', 'Kyoto': 'Japan',
    'Toronto': 'Canada', 'McGill': 'Canada',
    'Max Planck': 'Germany', 'Technical University of Munich': 'Germany',
    'National University of Singapore': 'Singapore',
    'Indian Institute of Technology': 'India'
}

def detect_country(inst):
    if inst == 'Unknown' or inst == 'Various': return 'Unknown'
    for key, country in country_map.items():
        if key.lower() in inst.lower():
            return country
    return 'Other'

# Apply extraction
institution_rankings['country'] = institution_rankings['entity_id'].apply(detect_country)

# Aggregate metrics by country
country_rankings = institution_rankings.groupby('country').agg({
    'paper_count': 'sum',
    'degree_centrality': 'mean',
    'source_diversity': 'sum'
}).reset_index()

# Sort and save
country_rankings = country_rankings.sort_values(by='paper_count', ascending=False)
country_rankings.to_csv('country_rankings.csv', index=False)

print('Country Intelligence Rankings generated.')
display(country_rankings.head(10))

Country Intelligence Rankings generated.


,country,paper_count,degree_centrality,source_diversity
5,Other,460,0.000562,318
4,Multiple,256,0.000526,149
8,USA,47,0.001889,8
6,Switzerland,7,0.002139,1
1,China,4,0.000698,3
7,UK,3,0.000546,2
2,Germany,3,0.000273,2
0,Canada,1,0.000319,1
3,Japan,1,0.000046,1


## Implement AI Research Assistant (RAG)

### Subtask:
Set up a simple Retrieval-Augmented Generation interface using the extracted topics and entities to allow natural language querying of the paper dataset.


**Reasoning**:
I will implement the retrieval component by using the SentenceTransformer library to encode queries and paper titles, allowing for semantic search.



In [13]:
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Load a pre-trained model for embeddings
print('Loading embedding model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Encode all paper titles to create a searchable index
paper_titles = df_clean['title'].tolist()
paper_embeddings = embed_model.encode(paper_titles, convert_to_tensor=True)

def retrieve_research_context(query, top_k=5):
    # Encode query
    query_embedding = embed_model.encode(query, convert_to_tensor=True)

    # Compute cosine similarity
    cos_scores = util.cos_sim(query_embedding, paper_embeddings)[0]

    # Get top-k results
    top_results = torch.topk(cos_scores, k=top_k)

    results = []
    for score, idx in zip(top_results[0], top_results[1]):
        paper_idx = int(idx)
        results.append({
            'title': df_clean.iloc[paper_idx]['title'],
            'authors': df_clean.iloc[paper_idx]['authors'],
            'year': df_clean.iloc[paper_idx]['year'],
            'score': float(score)
        })
    return results

# Test retrieval
test_query = 'What are the latest developments in blockchain for healthcare?'
context = retrieve_research_context(test_query)
print(f'Retrieved {len(context)} relevant papers for query: "{test_query}"')
for i, res in enumerate(context):
    print(f"{i+1}. [{res['year']}] {res['title']} (Score: {res['score']:.4f})")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieved 5 relevant papers for query: "What are the latest developments in blockchain for healthcare?"
1. [2016] Blockchain technology in healthcare: The revolution starts here (Score: 0.7959)
2. [2024] Blockchain Applications in Healthcare (Score: 0.7944)
3. [2020] Blockchain technology in healthcare: Challenges and opportunities (Score: 0.7908)
4. [2025] Advanced Topics in Blockchain for Healthcare (Score: 0.7758)
5. [2021] Blockchain technology applications in healthcare: An overview (Score: 0.7658)


**Reasoning**:
I will re-run the retrieval logic to ensure the variables are fully initialized in the kernel and then finalize the subtask by reporting the results.



In [14]:
from sentence_transformers import SentenceTransformer, util
import torch

# The model and embeddings were likely already partially processed.
# We re-verify the search function and ensure the context is printed clearly.

# 1. Load a pre-trained model for embeddings
print('Finalizing RAG retrieval component...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Encode all paper titles to create a searchable index
paper_titles = df_clean['title'].tolist()
paper_embeddings = embed_model.encode(paper_titles, convert_to_tensor=True)

def retrieve_research_context(query, top_k=5):
    query_embedding = embed_model.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, paper_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)

    results = []
    for score, idx in zip(top_results[0], top_results[1]):
        paper_idx = int(idx)
        results.append({
            'title': df_clean.iloc[paper_idx]['title'],
            'authors': df_clean.iloc[paper_idx]['authors'],
            'year': df_clean.iloc[paper_idx]['year'],
            'score': float(score)
        })
    return results

# Test retrieval with the target query
test_query = 'What are the latest developments in blockchain for healthcare?'
context = retrieve_research_context(test_query)
print(f'Retrieved {len(context)} relevant papers for query: "{test_query}"')
for i, res in enumerate(context):
    print(f"{i+1}. [{res['year']}] {res['title']} (Score: {res['score']:.4f})")

print('\nAI Research Assistant component is now ready for integration.')

Finalizing RAG retrieval component...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieved 5 relevant papers for query: "What are the latest developments in blockchain for healthcare?"
1. [2016] Blockchain technology in healthcare: The revolution starts here (Score: 0.7959)
2. [2024] Blockchain Applications in Healthcare (Score: 0.7944)
3. [2020] Blockchain technology in healthcare: Challenges and opportunities (Score: 0.7908)
4. [2025] Advanced Topics in Blockchain for Healthcare (Score: 0.7758)
5. [2021] Blockchain technology applications in healthcare: An overview (Score: 0.7658)

AI Research Assistant component is now ready for integration.


## Interactive Visualization Dashboard

### Subtask:
Create interactive network visualizations and data dashboards using PyVis and Plotly.


**Reasoning**:
I will start by creating the interactive knowledge graph visualization using PyVis, focusing on the most influential nodes to ensure the visualization is readable and performant.



In [15]:
from pyvis.network import Network
import pandas as pd
import IPython

# 1. Prepare data: Filter for top 150 nodes by degree centrality to avoid over-cluttering
top_nodes = graph_metrics_df.sort_values(by='degree_centrality', ascending=False).head(150)['entity_id'].tolist()

# Create a subgraph
sub_G = G.subgraph(top_nodes)

# 2. Initialize PyVis Network
net = Network(height='600px', width='100%', bgcolor='#222222', font_color='white', notebook=True, cdn_resources='remote')

# 3. Add nodes with styling based on type
color_map = {'Author': '#3498db', 'Institution': '#e74c3c', 'Concept': '#2ecc71', 'Paper': '#f1c40f'}

for node in sub_G.nodes():
    node_type = G.nodes[node].get('type', 'Unknown')
    net.add_node(node, label=node, color=color_map.get(node_type, '#95a5a6'), title=f'Type: {node_type}')

# 4. Add edges
for source, target in sub_G.edges():
    net.add_edge(source, target)

# 5. Generate and display
net.show('knowledge_graph.html')
print('Interactive Knowledge Graph generated as knowledge_graph.html')


knowledge_graph.html
Interactive Knowledge Graph generated as knowledge_graph.html


**Reasoning**:
I will use Plotly to create a bar chart for the top 10 emerging research topics based on growth rate and a horizontal bar chart for the top ranked authors by paper count.



In [16]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Visualize Research Trends (Top Emerging Topics)
top_trends = research_trends.sort_values(by='growth_rate', ascending=False).head(10)
fig_trends = px.bar(
    top_trends,
    x='growth_rate',
    y='Name',
    orientation='h',
    title='Top 10 Emerging Research Topics (Growth Rate Post-2020)',
    labels={'growth_rate': 'Growth Rate', 'Name': 'Topic Name'},
    color='growth_rate',
    color_continuous_scale='Viridis'
)
fig_trends.update_layout(yaxis={'categoryorder':'total ascending'}, height=500)
fig_trends.show()

# 2. Visualize Author Rankings
top_authors = author_rankings.head(10)
fig_authors = px.bar(
    top_authors,
    x='paper_count',
    y='entity_id',
    orientation='h',
    title='Top 10 Most Prolific Authors',
    labels={'paper_count': 'Number of Papers', 'entity_id': 'Author Name'},
    color='degree_centrality',
    color_continuous_scale='Reds'
)
fig_authors.update_layout(yaxis={'categoryorder':'total ascending'}, height=500)
fig_authors.show()

print('Plotly dashboards for trends and rankings have been generated.')

Plotly dashboards for trends and rankings have been generated.


## Consolidate to Multi-Sheet Excel

### Subtask:
Combine all generated CSV data and rankings into a single, comprehensive Excel workbook.


**Reasoning**:
I will use the pandas ExcelWriter to merge all the generated CSV files into a multi-sheet Excel workbook.



In [17]:
import pandas as pd

# Define the mapping of CSV files to sheet names
files_to_sheets = {
    'papers_raw.csv': 'Raw_Papers',
    'authors.csv': 'Author_Metadata',
    'institutions.csv': 'Institution_Metadata',
    'entities.csv': 'KG_Entities',
    'relationships.csv': 'KG_Relationships',
    'graph_metrics.csv': 'Graph_Metrics',
    'research_trends.csv': 'Research_Trends',
    'author_rankings.csv': 'Author_Rankings',
    'institution_rankings.csv': 'Institution_Rankings',
    'country_rankings.csv': 'Country_Rankings'
}

# Create the Excel workbook
with pd.ExcelWriter('global_science_knowledge_graph.xlsx', engine='openpyxl') as writer:
    for file_name, sheet_name in files_to_sheets.items():
        try:
            temp_df = pd.read_csv(file_name)
            temp_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f'Added {file_name} to sheet {sheet_name}.')
        except FileNotFoundError:
            print(f'Warning: {file_name} not found, skipping.')

print('\nComprehensive Excel workbook "global_science_knowledge_graph.xlsx" has been created successfully.')

Added papers_raw.csv to sheet Raw_Papers.
Added authors.csv to sheet Author_Metadata.
Added institutions.csv to sheet Institution_Metadata.
Added entities.csv to sheet KG_Entities.
Added relationships.csv to sheet KG_Relationships.
Added graph_metrics.csv to sheet Graph_Metrics.
Added research_trends.csv to sheet Research_Trends.
Added author_rankings.csv to sheet Author_Rankings.
Added institution_rankings.csv to sheet Institution_Rankings.
Added country_rankings.csv to sheet Country_Rankings.

Comprehensive Excel workbook "global_science_knowledge_graph.xlsx" has been created successfully.
